In [1]:
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GITHUB_PAT")
!rm -rf /kaggle/working/*
!git clone -q https://{tok}@github.com/vhrabar/SolarFilamentSegmentationChallenge.git

In [2]:
!cd SolarFilamentSegmentationChallenge && git checkout dev && git pull

Branch 'dev' set up to track remote branch 'dev' from 'origin'.
Switched to a new branch 'dev'
Already up to date.


## First: recalibrate the 0.34 checkpoint, no retraining

Weighted cross entropy does not put its decision point at a half. Minimising
`-[w y log p + (1-y) log(1-p)]` puts it at `w / (w + 1)`, so `pos_weight` 20 asks to be
thresholded at **0.952** — and the old grid stopped at 0.9. Every epoch after the seventh
picked 0.8 or 0.9, 25 of 40 picked 0.9 exactly: the search was running out of room, and the
masks came out systematically too fat, which is the most expensive error available on
structures eleven pixels thick.

Attach the previous run's output as an input dataset and point `--checkpoint` at it. This
costs minutes and isolates the threshold effect from everything else.

In [3]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --device cuda:0 calibrate \
    --checkpoint /kaggle/input/<previous-run-output>/runs/transunet_crop512/best.pt --tile 512 --overlap 0.25 --tile-batch 8 --workers 2

/bin/bash: line 1: previous-run-output: No such file or directory


## Then retrain: calibrated probabilities and averaged weights

Two changes, both aimed at what the history shows.

`--pos-weight 5` instead of the estimator's capped 20. Foreground-biased crops already lifted
the positive fraction from 0.43% to 1.9%, so the sampler is doing most of the balancing and a
weight of 20 on top squeezes every decision into the top 5% of the probability range. At 5 the
decision point sits at 0.83 with room on both sides. The validation grid is derived from
whatever `pos_weight` ends up being, so it can no longer miss the optimum.

`--swa-epochs 12` averages the weights of the final epochs. Validation PQ was flat from epoch 23
to 40 (0.31-0.36, no trend) with the learning rate decayed to 2e-6 — the run was wandering
around a minimum, not descending into one, which is exactly where averaging pays. It also
removes the selection bias of taking the best of 40 noisy epochs: val said 0.3586, the board
said 0.34.

In [4]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --seed 1 --device cuda:0 train \
    --arch transunet --crop 512 --crops-per-image 4 --foreground-prob 0.7 --pos-weight 5 \
    --batch-size 8 --epochs 40 --lr 2e-4 --dropout 0.1 --drop-path 0.1 --warmup-epochs 3 \
    --amp --workers 4 --swa-epochs 12 \
    --val-limit 64 --val-tile 512 --val-overlap 0.0 --tile-batch 8 \
    --output-dir /kaggle/working/runs/transunet_crop512_pw5

device: cuda:0
data: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
train 3696 crops / val 230 observations
validating at 0.084 0.199 0.404 0.648 0.833 0.931 0.974 0.990 0.996 (pos_weight 5.0 puts the decision point at 0.833)
TransUNet: 16.30M parameters
epoch 1/40  train 0.7932  val 0.4427  pq 0.0645 @ t=0.65  (306s)
  saved best.pt (pq 0.0645)
epoch 2/40  train 0.6814  val 0.2889  pq 0.1833 @ t=0.83  (301s)
  saved best.pt (pq 0.1833)
epoch 3/40  train 0.5906  val 0.1566  pq 0.2113 @ t=0.97  (303s)
  saved best.pt (pq 0.2113)
epoch 4/40  train 0.4956  val 0.0877  pq 0.2100 @ t=0.83  (307s)
epoch 5/40  train 0.4062  val 0.0635  pq 0.2382 @ t=0.83  (308s)
  saved best.pt (pq 0.2382)
epoch 6/40  train 0.3422  val 0.0226  pq 0.2145 @ t=0.65  (300s)
epoch 7/40  train 0.3034  val 0.0432  pq 0.1879 @ t=0.97  (301s)
epoch 8/40  train 0.2881  val 0.0190  pq 0.2165 @ t=0.83  (300s)
epoch 9/40  train 0.2783  val 0.0160  pq 0.2766 @ t=0.83  (298s)
  saved best.pt (

## Calibrate

Scored at 2048, the way the submission is. The grid comes from the checkpoint's `pos_weight`;
a winner that lands on a boundary is reported as a warning rather than accepted silently.
Use the whole validation split for the final calibration — 225 settings picked on a handful of
observations is the same selection bias as picking the best of 40 epochs.

In [5]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --device cuda:0 calibrate \
    --checkpoint /kaggle/working/runs/transunet_crop512_pw5/swa.pt --tile 512 --overlap 0.25 --tile-batch 8 --workers 2

device: cuda:0
data: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
calibrating on 230/230 validation observations at 2048, model input native, tile 512
 threshold  offset  min_area      pq
     0.648      +0       512  0.3919
     0.404      +0       512  0.3871
     0.199      -1       512  0.3865
     0.404      -1       512  0.3863
     0.833      +0       512  0.3823
     0.199      +0       512  0.3794
     0.648      +0       256  0.3769
     0.199      -1       256  0.3768
     0.833      +0       256  0.3751
     0.404      -1       256  0.3745
best pq 0.3919 at threshold 0.648, offset +0, min_area 512
stored in /kaggle/working/runs/transunet_crop512_pw5/swa.pt and /kaggle/working/runs/transunet_crop512_pw5/swa.calibration.json


## Predict

The calibrated threshold, offset and minimum area come out of the checkpoint.

In [6]:
!cd /kaggle/working/SolarFilamentSegmentationChallenge && python -m src.main --device cuda:0 predict \
    --checkpoint /kaggle/working/runs/transunet_crop512_pw5/swa.pt --output-dir /kaggle/working/runs/transunet_crop512_pw5/predictions \
    --batch-size 1 --workers 4 --tile 512 --overlap 0.25 --tile-batch 8

device: cuda:0
data: /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
checkpoint epoch 12, size native, tile 512, threshold 0.65, offset +0, min_area 512
180 test observations
predict: 100%|████████████████████████████████| 180/180 [12:13<00:00,  4.08s/it]
1491 filaments over 178/180 observations -> /kaggle/working/runs/transunet_crop512_pw5/predictions/submission.csv
